In [ ]:
from google.colab import files
files.upload()

Saving resnet34_cifar10_baseline_final.pth to resnet34_cifar10_baseline_final.pth


In [1]:

import argparse
import copy
import os
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

try:
    from thop import profile
except Exception:
    profile = None


# ============================================================
# Kernel-Level Pruning Algorithm (real kernel removal)
# ============================================================
# This script implements professor-style kernel pruning:
#   1) For each conv layer: treat each (out_channel, in_channel) kernel as one vector
#   2) Select low-norm kernels (bottom norm_percent)
#   3) Build similarity groups among low-norm kernels with cosine > tau_s
#   4) Keep strongest kernel per group, remove redundant kernels
#   5) Replace Conv2d with compact kernel-sparse conv modules
#   6) Fine-tune full model for recovery
#
# Note:
#   The model keeps the same feature-map interface, but each pruned conv stores
#   only the retained (out_channel, in_channel) kernels, so parameter count drops.
# ============================================================


# Utility: reproducibility across runs.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Utility: rebuild architecture skeleton and classifier head from checkpoint metadata.
def replace_classifier(model: nn.Module, arch: str, num_classes: int) -> nn.Module:
    arch = arch.lower()
    if arch in {"resnet18", "resnet34"}:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == "densenet121":
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")
    return model


# Utility: model factory for reusable dataset/backbone combinations.
def build_model(arch: str, num_classes: int) -> nn.Module:
    arch = arch.lower()
    if arch == "resnet18":
        model = models.resnet18(weights=None)
    elif arch == "resnet34":
        model = models.resnet34(weights=None)
    elif arch == "densenet121":
        model = models.densenet121(weights=None)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")
    return replace_classifier(model, arch, num_classes)


# Data loader helper: currently CIFAR-10 default; reusable for CIFAR-100.
def get_cifar_loaders(
    data_root: str,
    dataset_name: str,
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
) -> Tuple[DataLoader, DataLoader, int]:
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    train_tf = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    test_tf = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )

    name = dataset_name.lower()
    if name == "cifar10":
        train_ds = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tf)
        test_ds = datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tf)
        num_classes = 10
    elif name == "cifar100":
        train_ds = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_tf)
        test_ds = datasets.CIFAR100(root=data_root, train=False, download=True, transform=test_tf)
        num_classes = 100
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )
    return train_loader, test_loader, num_classes


# Utility: top-k correctness.
def topk_correct(outputs: torch.Tensor, targets: torch.Tensor, k: int = 5) -> int:
    _, pred = outputs.topk(k, dim=1)
    return pred.eq(targets.view(-1, 1)).sum().item()


# Metrics: loss + top-1 + top-5 on a loader.
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    total = 0
    total_loss = 0.0
    top1 = 0
    top5 = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        total += y.size(0)
        total_loss += loss.item() * y.size(0)
        top1 += out.argmax(dim=1).eq(y).sum().item()
        top5 += topk_correct(out, y, k=5)
    return {"loss": total_loss / max(total, 1), "top1": 100.0 * top1 / max(total, 1), "top5": 100.0 * top5 / max(total, 1)}


# Metrics: parameter count.
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


# Metrics: FLOPs estimate (optional/slow).
@torch.no_grad()
def compute_flops(model: nn.Module, device: torch.device) -> float:
    dummy = torch.randn(1, 3, 224, 224, device=device)
    # 1) Try THOP first when available.
    if profile is not None:
        try:
            flops, _ = profile(model, inputs=(dummy,), verbose=False)
            return float(flops)
        except Exception:
            pass

    # 2) Fallback: analytic hook-based FLOPs for Conv/Linear/custom pruned conv.
    flops_total = {"value": 0.0}
    hooks = []

    def conv_hook(module: nn.Conv2d, inputs: Tuple[torch.Tensor], output: torch.Tensor) -> None:
        x = inputs[0]
        n = x.size(0)
        out_h, out_w = output.size(2), output.size(3)
        k_h, k_w = module.kernel_size
        in_per_group = module.in_channels // module.groups
        macs = n * module.out_channels * out_h * out_w * in_per_group * k_h * k_w
        flops_total["value"] += float(macs)

    def pruned_conv_hook(module: "PrunedKernelConv2d", inputs: Tuple[torch.Tensor], output: torch.Tensor) -> None:
        x = inputs[0]
        n = x.size(0)
        out_h, out_w = output.size(2), output.size(3)
        # Kept kernels x kernel elements at each spatial location.
        macs = n * out_h * out_w * module.num_kept_kernels * module.kernel_elements
        flops_total["value"] += float(macs)

    def linear_hook(module: nn.Linear, inputs: Tuple[torch.Tensor], output: torch.Tensor) -> None:
        x = inputs[0]
        n = x.size(0)
        macs = n * module.in_features * module.out_features
        flops_total["value"] += float(macs)

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, PrunedKernelConv2d):
            hooks.append(m.register_forward_hook(pruned_conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))

    was_training = model.training
    model.eval()
    _ = model(dummy)
    if was_training:
        model.train()

    for h in hooks:
        h.remove()

    return float(flops_total["value"])


class PrunedKernelConv2d(nn.Module):
    """Compact conv storing only kept (out_channel, in_channel) kernels."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: Tuple[int, int],
        stride: Tuple[int, int],
        padding: Tuple[int, int],
        dilation: Tuple[int, int],
        bias: Optional[torch.Tensor],
        out_idx: torch.Tensor,
        in_idx: torch.Tensor,
        compact_weight: torch.Tensor,
    ) -> None:
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.kernel_size = (int(kernel_size[0]), int(kernel_size[1]))
        self.stride = (int(stride[0]), int(stride[1]))
        self.padding = (int(padding[0]), int(padding[1]))
        self.dilation = (int(dilation[0]), int(dilation[1]))
        self.kernel_elements = self.kernel_size[0] * self.kernel_size[1]

        self.register_buffer("out_idx", out_idx.long())
        self.register_buffer("in_idx", in_idx.long())
        self.compact_weight = nn.Parameter(compact_weight)
        if bias is None:
            self.bias = None
        else:
            self.bias = nn.Parameter(bias.clone())

    @property
    def num_kept_kernels(self) -> int:
        return int(self.compact_weight.size(0))

    @staticmethod
    def from_conv(conv: nn.Conv2d, keep_idx: torch.Tensor) -> "PrunedKernelConv2d":
        if conv.groups != 1:
            raise ValueError("PrunedKernelConv2d supports groups=1 only.")
        if conv.padding_mode != "zeros":
            raise ValueError("PrunedKernelConv2d supports zero padding mode only.")

        cout, cin, kh, kw = conv.weight.shape
        keep_idx = keep_idx.long().to(conv.weight.device)
        out_idx = keep_idx // cin
        in_idx = keep_idx % cin
        compact_weight = conv.weight.data.view(cout * cin, kh, kw)[keep_idx].clone()
        bias = conv.bias.data if conv.bias is not None else None

        return PrunedKernelConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=(kh, kw),
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            bias=bias,
            out_idx=out_idx,
            in_idx=in_idx,
            compact_weight=compact_weight,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Memory-safe execution:
        # reconstruct dense kernel tensor from compact kept kernels, then use cuDNN conv.
        kh, kw = self.kernel_size
        full_w = x.new_zeros((self.out_channels, self.in_channels, kh, kw))
        if self.num_kept_kernels > 0:
            full_w[self.out_idx, self.in_idx, :, :] = self.compact_weight
        return F.conv2d(
            x,
            full_w,
            self.bias,
            stride=self.stride,
            padding=self.padding,
            dilation=self.dilation,
        )


def replace_module_by_name(model: nn.Module, module_name: str, new_module: nn.Module) -> None:
    parent_name, _, child_name = module_name.rpartition(".")
    parent = model.get_submodule(parent_name) if parent_name else model
    if child_name.isdigit() and isinstance(parent, (nn.Sequential, nn.ModuleList)):
        parent[int(child_name)] = new_module
    else:
        setattr(parent, child_name, new_module)


# Utility: get conv layer names to enforce layer-wise pruning order.
def get_conv_layer_names(model: nn.Module) -> List[str]:
    names = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            names.append(name)
    return names


# Utility: connected components from similarity graph.
def _connected_groups(sim: torch.Tensor, threshold: float) -> List[List[int]]:
    n = sim.size(0)
    if n < 2:
        return []

    parent = list(range(n))

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    edges = torch.nonzero(torch.triu(sim > threshold, diagonal=1), as_tuple=False).tolist()
    for i, j in edges:
        union(i, j)

    groups: Dict[int, List[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return [g for g in groups.values() if len(g) > 1]


# Phase 2 core helper:
# For one conv layer, compute redundant kernel indices based on:
#   - cosine similarity threshold
#   - low-norm kernel selection
def find_redundant_kernels(
    conv: nn.Conv2d,
    tau_s: float,
    norm_percent: float,
    max_candidates: int,
    min_keep: int,
) -> Dict[str, object]:
    w = conv.weight.data  # [Cout, Cin, k, k]
    cout, cin, _, _ = w.shape
    kernels = w.view(cout * cin, -1)
    total_kernels = int(cout * cin)
    if total_kernels <= min_keep:
        return {
            "prune_idx": [],
            "num_groups": 0,
            "num_candidates": 0,
            "grouped_kernels": 0,
            "kernels_before": total_kernels,
            "out_channels": int(cout),
            "in_channels": int(cin),
        }

    norms = torch.norm(kernels, p=2, dim=1)
    tau_n = torch.quantile(norms, norm_percent / 100.0)
    cand_idx = torch.where(norms <= tau_n)[0]

    if cand_idx.numel() > max_candidates:
        keep_local = torch.argsort(norms[cand_idx])[:max_candidates]
        cand_idx = cand_idx[keep_local]

    if cand_idx.numel() < 2:
        return {
            "prune_idx": [],
            "num_groups": 0,
            "num_candidates": int(cand_idx.numel()),
            "grouped_kernels": 0,
            "kernels_before": total_kernels,
            "out_channels": int(cout),
            "in_channels": int(cin),
        }

    x = kernels[cand_idx]
    x = x / (torch.norm(x, p=2, dim=1, keepdim=True) + 1e-12)
    sim = x @ x.T
    groups = _connected_groups(sim, tau_s)

    prune_idx: List[int] = []
    grouped_kernels = 0
    for g in groups:
        ids = cand_idx[torch.tensor(g, device=cand_idx.device)]
        # Keep strongest (highest norm), prune other redundant kernels.
        local_norms = norms[ids]
        keep = ids[torch.argmax(local_norms)]
        for idx in ids:
            if int(idx.item()) != int(keep.item()):
                prune_idx.append(int(idx.item()))
        grouped_kernels += len(g)

    prune_idx = sorted(set(prune_idx))
    # Keep at least min_keep kernels in this layer.
    max_prunable = max(0, total_kernels - min_keep)
    if len(prune_idx) > max_prunable:
        prune_idx = prune_idx[:max_prunable]

    return {
        "prune_idx": prune_idx,
        "num_groups": len(groups),
        "num_candidates": int(cand_idx.numel()),
        "grouped_kernels": int(grouped_kernels),
        "kernels_before": total_kernels,
        "out_channels": int(cout),
        "in_channels": int(cin),
    }


# Phase 2 kernel pruning:
# Performs layer-wise real kernel pruning by replacing Conv2d with compact modules.
def kernel_prune_model(
    model: nn.Module,
    example_inputs: torch.Tensor,
    tau_s: float = 0.55,
    norm_percent: float = 85.0,
    max_candidates_per_layer: int = 10000,
    min_keep_channels: int = 1,
) -> Dict[str, object]:
    kernel_layer_stats: List[Dict[str, object]] = []
    total_pruned_kernels = 0
    total_kernels_before = 0

    conv_names = get_conv_layer_names(model)
    for li, name in enumerate(conv_names):
        named = dict(model.named_modules())
        if name not in named:
            continue
        conv = named[name]
        if not isinstance(conv, nn.Conv2d):
            continue

        print(f"[Kernel Pruning] Layer {li+1}/{len(conv_names)}: {name}", flush=True)
        info = find_redundant_kernels(
            conv=conv,
            tau_s=tau_s,
            norm_percent=norm_percent,
            max_candidates=max_candidates_per_layer,
            min_keep=min_keep_channels,
        )
        print(
            f"Layer: {name} | candidates={info['num_candidates']} | "
            f"groups={info['num_groups']} | pruned={len(info['prune_idx'])}",
            flush=True
            )

        prune_idx = info["prune_idx"]
        pruned = 0
        if len(prune_idx) > 0:
            kernels_before = int(info["kernels_before"])
            keep_mask = torch.ones(kernels_before, dtype=torch.bool, device=conv.weight.device)
            keep_mask[torch.tensor(prune_idx, device=conv.weight.device)] = False
            keep_idx = torch.where(keep_mask)[0]
            compact_conv = PrunedKernelConv2d.from_conv(conv, keep_idx)
            replace_module_by_name(model, name, compact_conv.to(conv.weight.device))
            pruned = len(prune_idx)

        total_pruned_kernels += pruned
        total_kernels_before += int(info["kernels_before"])
        kernel_layer_stats.append(
            {
                "layer_index": li,
                "layer_name": name,
                "out_channels": info["out_channels"],
                "in_channels": info["in_channels"],
                "kernels_before": info["kernels_before"],
                "kernels_after": info["kernels_before"] - pruned,
                "num_candidates": info["num_candidates"],
                "num_groups": info["num_groups"],
                "grouped_kernels": info["grouped_kernels"],
                "pruned_kernels": pruned,
            }
        )
    _ = example_inputs
    kernel_prune_pct = 100.0 * total_pruned_kernels / max(total_kernels_before, 1)
    return {
        "total_kernels_before": int(total_kernels_before),
        "total_pruned_kernels": int(total_pruned_kernels),
        "kernel_prune_pct": float(kernel_prune_pct),
        "kernel_layer_stats": kernel_layer_stats,
    }


# Phase 4 fine-tuning: supports full fine-tuning and optional freeze-early mode.
def fine_tune(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
    epochs: int = 20,
    lr: float = 1e-4,
    amp: bool = True,
) -> List[Dict[str, float]]:
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    use_amp = amp and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    history: List[Dict[str, float]] = []
    for ep in range(epochs):
        model.train()
        total = 0
        top1 = 0
        run_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total += y.size(0)
            run_loss += loss.item() * y.size(0)
            top1 += out.argmax(dim=1).eq(y).sum().item()

        scheduler.step()
        test_m = evaluate(model, test_loader, criterion, device)
        row = {
            "epoch": ep + 1,
            "train_loss": run_loss / max(total, 1),
            "train_top1": 100.0 * top1 / max(total, 1),
            "test_loss": test_m["loss"],
            "test_top1": test_m["top1"],
            "test_top5": test_m["top5"],
        }
        history.append(row)
        print(
            f"Epoch {ep+1:02d}/{epochs} | "
            f"Train Loss: {row['train_loss']:.4f} | Train Top1: {row['train_top1']:.2f} | "
            f"Test Loss: {row['test_loss']:.4f} | Test Top1: {row['test_top1']:.2f}",
            flush=True,
        )
    return history


# Utility: load baseline checkpoint and clean potential THOP metadata keys.
def load_phase1_model(
    checkpoint_path: str,
    device: torch.device,
    arch_override: Optional[str] = None,
    num_classes_override: Optional[int] = None,
) -> Tuple[nn.Module, Dict]:
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device)
    arch = (arch_override or ckpt.get("arch", "resnet34")).lower()
    num_classes = int(num_classes_override or ckpt.get("num_classes", 10))
    model = build_model(arch, num_classes).to(device)

    raw_state = ckpt["model_state_dict"]
    clean_state = {k: v for k, v in raw_state.items() if "total_ops" not in k and "total_params" not in k}
    model.load_state_dict(clean_state, strict=True)
    return model, ckpt


# CLI: defaults target CIFAR-10, tau_s=0.85, norm=30, 20 epochs. Use parse_known_args for Colab/Jupyter.
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Kernel-level pruning (masking), single-run")

    parser.add_argument("--checkpoint", type=str, required=True, help="Path to baseline .pth")
    parser.add_argument("--data_root", type=str, default="./data")
    parser.add_argument("--dataset", type=str, default="cifar10", choices=["cifar10", "cifar100"])
    parser.add_argument("--output_dir", type=str, default="artifacts/phase2_3_4_structural")
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--tau_s", type=float, default=0.65, help="Single-run similarity threshold")
    parser.add_argument("--norm_percent", type=float, default= 70.0, help="Single-run norm percentile (bottom %)")
    parser.add_argument("--max_candidates_per_layer", type=int, default= 10000)
    parser.add_argument(
        "--min_keep_channels",
        type=int,
        default=1,
        help="Minimum number of kernels kept per conv layer (kept arg name for backward compatibility)",
    )
    parser.add_argument("--amp", action="store_true", help="Mixed precision fine-tuning on CUDA")
    parser.add_argument("--skip_flops", action="store_true", help="Skip THOP FLOPs (faster)")
    parser.add_argument("--seed", type=int, default=42)

    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    device = torch.device(args.device)
    pin_memory = device.type == "cuda"

    os.makedirs(args.output_dir, exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, "tables"), exist_ok=True)

    print("\n=== Phase 1 Handoff: Load baseline checkpoint ===", flush=True)
    baseline_model, ckpt = load_phase1_model(args.checkpoint, device)
    arch = ckpt.get("arch", "unknown").lower()
    if arch not in {"resnet18", "resnet34", "densenet121"}:
        raise ValueError(f"Unsupported arch from checkpoint: {arch}")

    print(f"\n=== Data: {args.dataset.upper()} loaders ===", flush=True)
    train_loader, test_loader, num_classes = get_cifar_loaders(
        data_root=args.data_root,
        dataset_name=args.dataset,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        pin_memory=pin_memory,
    )
    _ = num_classes
    criterion = nn.CrossEntropyLoss()

    baseline_metrics = evaluate(baseline_model, test_loader, criterion, device)
    baseline_params = count_params(baseline_model)
    baseline_flops = float("nan") if args.skip_flops else compute_flops(baseline_model, device)
    baseline_ckpt_path = os.path.join(args.output_dir, "checkpoints", f"{arch}_{args.dataset}_baseline_snapshot.pth")
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "baseline",
            "model_state_dict": baseline_model.state_dict(),
            "metrics": baseline_metrics,
            "params": baseline_params,
            "flops": baseline_flops,
        },
        baseline_ckpt_path,
    )

    print("\n=== Phase 2+3: Layer-wise Kernel Pruning (Masking) ===", flush=True)
    pruned_model = copy.deepcopy(baseline_model).to(device)
    example_inputs = torch.randn(1, 3, 224, 224, device=device)
    prune_info = kernel_prune_model(
        model=pruned_model,
        example_inputs=example_inputs,
        tau_s=args.tau_s,
        norm_percent=args.norm_percent,
        max_candidates_per_layer=args.max_candidates_per_layer,
        min_keep_channels=args.min_keep_channels,
    )
    post_prune_metrics = evaluate(pruned_model, test_loader, criterion, device)
    post_prune_params = count_params(pruned_model)
    post_prune_flops = float("nan") if args.skip_flops else compute_flops(pruned_model, device)
    param_reduction_pct = 100.0 * (baseline_params - post_prune_params) / max(baseline_params, 1)
    flops_reduction_pct = (
        np.nan
        if (args.skip_flops or np.isnan(baseline_flops) or np.isnan(post_prune_flops) or baseline_flops == 0)
        else 100.0 * (baseline_flops - post_prune_flops) / baseline_flops
    )

    print("\n=== Baseline vs Post-Prune (Before FT) ===", flush=True)
    print(
        f"Baseline   -> Loss: {baseline_metrics['loss']:.6f}, Top1: {baseline_metrics['top1']:.4f}, Top5: {baseline_metrics['top5']:.4f}",
        flush=True,
    )
    print(
        f"PostPrune  -> Loss: {post_prune_metrics['loss']:.6f}, Top1: {post_prune_metrics['top1']:.4f}, Top5: {post_prune_metrics['top5']:.4f}",
        flush=True,
    )
    print(
        f"Drop/Incr  -> Top1 drop: {baseline_metrics['top1'] - post_prune_metrics['top1']:.4f}, "
        f"Loss increase: {post_prune_metrics['loss'] - baseline_metrics['loss']:.6f}",
        flush=True,
    )
    print(
        f"Total kernels before pruning: {prune_info['total_kernels_before']} | "
        f"Total pruned kernels: {prune_info['total_pruned_kernels']}",
        flush=True,
    )
    print(
        f"Kernel prune %: {prune_info['kernel_prune_pct']:.4f} | "
        f"Param reduction % (structural): {param_reduction_pct:.4f} | "
        f"FLOPs reduction % (structural): {flops_reduction_pct if not np.isnan(flops_reduction_pct) else float('nan'):.4f}",
        flush=True,
    )

    post_prune_ckpt_path = os.path.join(args.output_dir, "checkpoints", f"{arch}_{args.dataset}_post_prune_pre_ft.pth")
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "post_prune_pre_ft",
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "model_state_dict": pruned_model.state_dict(),
            "metrics": post_prune_metrics,
            "prune_info": prune_info,
            "params": post_prune_params,
            "flops": post_prune_flops,
        },
        post_prune_ckpt_path,
    )

    print("\n=== Phase 4: Fine-tuning (full) ===", flush=True)
    ft_model = copy.deepcopy(pruned_model).to(device)
    ft_history = fine_tune(
        model=ft_model,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=args.epochs,
        lr=args.lr,
        amp=args.amp,
    )
    post_ft_metrics = evaluate(ft_model, test_loader, criterion, device)
    post_ft_params = count_params(ft_model)
    ft_flops = float("nan") if args.skip_flops else compute_flops(ft_model, device)

    post_ft_ckpt_path = os.path.join(
        args.output_dir, "checkpoints", f"{arch}_{args.dataset}_post_prune_post_ft_full.pth"
    )
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "post_prune_post_ft_full",
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "model_state_dict": ft_model.state_dict(),
            "metrics": post_ft_metrics,
            "prune_info": prune_info,
            "finetune_history": ft_history,
            "params": post_ft_params,
            "flops": ft_flops,
        },
        post_ft_ckpt_path,
    )
    ft_history_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_finetune_history_full.csv")
    pd.DataFrame(ft_history).to_csv(ft_history_path, index=False)

    layer_stats_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_layer_stats_kernel.csv")
    pd.DataFrame(prune_info["kernel_layer_stats"]).to_csv(layer_stats_path, index=False)
    # Backward-compatible alias expected by earlier artifacts list.
    layer_stats_legacy_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_layer_stats.csv")
    pd.DataFrame(prune_info["kernel_layer_stats"]).to_csv(layer_stats_legacy_path, index=False)

    metrics_rows = [
        {
            "stage": "baseline",
            "top1": baseline_metrics["top1"],
            "top5": baseline_metrics["top5"],
            "loss": baseline_metrics["loss"],
            "params": baseline_params,
            "flops": baseline_flops,
            "tau_s": np.nan,
            "norm_percent": np.nan,
            "total_kernels_before": prune_info["total_kernels_before"],
            "pruned_kernels_total": 0,
            "kernel_prune_%": 0.0,
            "param_reduction_%": 0.0,
            "flops_reduction_%": 0.0 if not np.isnan(baseline_flops) else np.nan,
            "checkpoint_path": baseline_ckpt_path,
        },
        {
            "stage": "post_prune_pre_ft",
            "top1": post_prune_metrics["top1"],
            "top5": post_prune_metrics["top5"],
            "loss": post_prune_metrics["loss"],
            "params": post_prune_params,
            "flops": post_prune_flops,
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "total_kernels_before": prune_info["total_kernels_before"],
            "pruned_kernels_total": prune_info["total_pruned_kernels"],
            "kernel_prune_%": prune_info["kernel_prune_pct"],
            "param_reduction_%": param_reduction_pct,
            "flops_reduction_%": flops_reduction_pct,
            "checkpoint_path": post_prune_ckpt_path,
        },
        {
            "stage": "post_prune_post_ft_full",
            "top1": post_ft_metrics["top1"],
            "top5": post_ft_metrics["top5"],
            "loss": post_ft_metrics["loss"],
            "params": post_ft_params,
            "flops": ft_flops,
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "total_kernels_before": prune_info["total_kernels_before"],
            "pruned_kernels_total": prune_info["total_pruned_kernels"],
            "kernel_prune_%": prune_info["kernel_prune_pct"],
            "param_reduction_%": 100.0 * (baseline_params - post_ft_params) / max(baseline_params, 1),
            "flops_reduction_%": (
                np.nan
                if (args.skip_flops or np.isnan(baseline_flops) or np.isnan(ft_flops) or baseline_flops == 0)
                else 100.0 * (baseline_flops - ft_flops) / baseline_flops
            ),
            "checkpoint_path": post_ft_ckpt_path,
        },
    ]

    # Required report fields: drop after pruning and recovery after fine-tuning.
    for row in metrics_rows:
        stage = row["stage"]
        row["acc_drop_after_prune"] = baseline_metrics["top1"] - post_prune_metrics["top1"]
        row["loss_increase_after_prune"] = post_prune_metrics["loss"] - baseline_metrics["loss"]
        if stage == "post_prune_post_ft_full":
            row["acc_recovery_after_ft"] = row["top1"] - post_prune_metrics["top1"]
            row["loss_recovery_after_ft"] = post_prune_metrics["loss"] - row["loss"]
        else:
            row["acc_recovery_after_ft"] = np.nan
            row["loss_recovery_after_ft"] = np.nan

    metrics_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_metrics_table.csv")
    pd.DataFrame(metrics_rows).to_csv(metrics_path, index=False)

    print("\n=== Drop/Recovery Summary ===", flush=True)
    print(
        f"After pruning -> Top1 drop: {baseline_metrics['top1'] - post_prune_metrics['top1']:.4f}, "
        f"Loss increase: {post_prune_metrics['loss'] - baseline_metrics['loss']:.6f}",
        flush=True,
    )
    print(
        f"After FT (full) -> Top1 recovery: {post_ft_metrics['top1'] - post_prune_metrics['top1']:.4f}, "
        f"Loss recovery: {post_prune_metrics['loss'] - post_ft_metrics['loss']:.6f}",
        flush=True,
    )
    print(
        f"Kernel prune %: {prune_info['kernel_prune_pct']:.4f} | "
        f"Param reduction %: {param_reduction_pct:.4f} | "
        f"FLOPs reduction %: {flops_reduction_pct if not np.isnan(flops_reduction_pct) else float('nan'):.4f}",
        flush=True,
    )

    print("\nDone. Saved artifacts:", flush=True)
    print(f"- {baseline_ckpt_path}", flush=True)
    print(f"- {post_prune_ckpt_path}", flush=True)
    print(f"- {post_ft_ckpt_path}", flush=True)
    print(f"- {layer_stats_path}", flush=True)
    print(f"- {layer_stats_legacy_path}", flush=True)
    print(f"- {metrics_path}", flush=True)
    print(f"- {ft_history_path}", flush=True)



In [2]:
import sys

# Save the original sys.argv
original_argv = sys.argv

# Simulate command-line arguments to provide the required --checkpoint
sys.argv = ['main.py', '--checkpoint', 'resnet34_cifar10_baseline_final.pth']

try:
    main()
finally:
    # Restore sys.argv to its original state to avoid interfering with other cells
    sys.argv = original_argv


=== Phase 1 Handoff: Load baseline checkpoint ===

=== Data: CIFAR10 loaders ===


100%|██████████| 170M/170M [00:13<00:00, 12.3MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ratio


=== Phase 2+3: Layer-wise Kernel Pruning (Masking) ===
[Kernel Pruning] Layer 1/36: conv1
Layer: conv1 | candidates=134 | groups=11 | pruned=106
[Kernel Pruning] Layer 2/36: layer1.0.conv1
Layer: layer1.0.conv1 | candidates=2867 | groups=1 | pruned=2866
[Kernel Pruning] Layer 3/36: layer1.0.conv2
Layer: layer1.0.conv2 | candidates=2867 | groups=1 | pruned=2866
[Kernel Pruning] Layer 4/36: layer1.1.conv1
Layer: layer1.1.conv1 | candidates=2867 | groups=1 | pruned=2866
[Kernel Pruning] Layer 5/36: layer1.1.conv2
Layer: layer1.1.conv2 | candidates=2867 | groups=1 | pruned=2866
[Kernel Pruning] Layer 6/36: layer1.2.conv1
Layer: layer1.2.conv1 | candidates=2867 | groups=1 | pruned=2866
[Kernel Pruning] Layer 7/36: layer1.2.conv2
Layer: layer1.2.conv2 | candidates=2867 | groups=1 | pruned=2866
[Kernel Pruning] Layer 8/36: layer2.0.conv1
Layer: layer2.0.conv1 | candidates=5734 | groups=1 | pruned=5733
[Kernel Pruning] Layer 9/36: layer2.0.conv2
Layer: layer2.0.conv2 | candidates=10000 | grou

/tmp/ipykernel_1471/3697526094.py:516: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_1471/3697526094.py:527: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 01/20 | Train Loss: 0.1751 | Train Top1: 93.99 | Test Loss: 0.1633 | Test Top1: 94.48
Epoch 02/20 | Train Loss: 0.0924 | Train Top1: 96.87 | Test Loss: 0.1689 | Test Top1: 94.70
Epoch 03/20 | Train Loss: 0.0725 | Train Top1: 97.58 | Test Loss: 0.1590 | Test Top1: 94.85
Epoch 04/20 | Train Loss: 0.0579 | Train Top1: 98.07 | Test Loss: 0.1451 | Test Top1: 95.43
Epoch 05/20 | Train Loss: 0.0438 | Train Top1: 98.50 | Test Loss: 0.1533 | Test Top1: 95.36
Epoch 06/20 | Train Loss: 0.0332 | Train Top1: 98.88 | Test Loss: 0.1580 | Test Top1: 95.52
Epoch 07/20 | Train Loss: 0.0286 | Train Top1: 99.02 | Test Loss: 0.1589 | Test Top1: 95.54
Epoch 08/20 | Train Loss: 0.0243 | Train Top1: 99.20 | Test Loss: 0.1517 | Test Top1: 95.79
Epoch 09/20 | Train Loss: 0.0184 | Train Top1: 99.43 | Test Loss: 0.1426 | Test Top1: 95.94
Epoch 10/20 | Train Loss: 0.0119 | Train Top1: 99.63 | Test Loss: 0.1535 | Test Top1: 96.01
Epoch 11/20 | Train Loss: 0.0092 | Train Top1: 99.72 | Test Loss: 0.1469 | Test 